In [1]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [2]:
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_squared_error, r2_score,
                              accuracy_score, classification_report,
                              confusion_matrix, silhouette_score)
from sklearn.pipeline import Pipeline

In [1]:
!python -m pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# STYLE
PALETTE = ["#2D6A4F", "#52B788", "#B7E4C7", "#D4A373", "#E76F51", "#264653"]
sns.set_theme(style="darkgrid", palette=PALETTE)
plt.rcParams.update({"figure.dpi": 130, "font.family": "DejaVu Sans"})


In [8]:
# 1. LOAD
print("\n" + "="*60)
print("  STEP 1 — LOAD DATA")
print("="*60)
df = pd.read_csv("c:\\Users\\hp\\Downloads\\coffee-listings-from-all-walmart-stores.csv")
print(f"Shape  : {df.shape}")
print(df.head(3).to_string())


  STEP 1 — LOAD DATA
Shape  : (1400, 9)
                                                                               title                 coffee_type  rating  reviews  seller_name                                                                                                                                              thumbnail  price      weight  weight_formatted_to_gramms
0                                    folgers classic roast ground coffee, 40.3-ounce               classic roast     3.8       93  walmart.com    https://i5.walmartimages.com/asr/1fbbd523-8554-4a85-8107-2568a125a6d2.2395bc5a9e08e45dc51e62b268776b65.jpeg?odnHeight=180&odnWidth=180&odnBg=FFFFFF  13.92  40.3-ounce                      1142.5
1  café bustelo, espresso style dark roast ground coffee, vacuum-packed 10 oz. brick         espresso,dark roast     4.7      914  walmart.com  https://i5.walmartimages.com/asr/99a53df0-0471-4b63-abe7-d6b8d314ebec_1.bbbb1a66318deade7f16115f60ac8725.jpeg?odnHeight=180&odnWidth=

In [26]:
# 2. CLEANING
print("\n" + "="*60)
print("  STEP 2 — DATA CLEANING")
print("="*60)
print("Missing values before cleaning:")
print(df.isnull().sum())
# Replace 0 price with NaN
df["price"] = df["price"].replace(0, np.nan)



  STEP 2 — DATA CLEANING
Missing values before cleaning:
title                           0
coffee_type                   265
rating                          0
reviews                         0
seller_name                     0
thumbnail                       0
price                          57
weight                          0
weight_formatted_to_gramms      0
price_per_gram                  0
log_reviews                     0
is_popular                      0
dtype: int64


In [83]:
# Replace 0 price with NaN
df["price"] = df["price"].replace(0, np.nan)

In [28]:
# Drop rows missing weight
df = df.dropna(subset=["weight_formatted_to_gramms"]).copy()

In [29]:
# Fill missing values safely
df["price"]  = df["price"].fillna(df["price"].median())
df["rating"] = df["rating"].fillna(df["rating"].median())
df["reviews"] = df["reviews"].fillna(0)
df["coffee_type"] = df["coffee_type"].fillna("unknown")


In [30]:
# Remove clear non-coffee items
df = df[~((df["rating"]==0) & (df["reviews"]==0) & (df["price"] > 20))].copy()
df.reset_index(drop=True, inplace=True)


In [33]:
# Feature engineering — avoid inplace on columns
df["weight_formatted_to_gramms"] = df["weight_formatted_to_gramms"].replace(0, np.nan)
df["price_per_gram"] = df["price"] / df["weight_formatted_to_gramms"]
df["price_per_gram"] = df["price_per_gram"].replace([np.inf, -np.inf], np.nan)
df["price_per_gram"] = df["price_per_gram"].fillna(df["price_per_gram"].median())
df["weight_formatted_to_gramms"] = df["weight_formatted_to_gramms"].fillna(df["weight_formatted_to_gramms"].median())

df["log_reviews"] = np.log1p(df["reviews"])
df["is_popular"] = (df["reviews"] >= df["reviews"].quantile(0.75)).astype(int)

In [34]:
def extract_roast(ct):
    ct = str(ct).lower()
    if "dark" in ct: return "dark"
    if "medium" in ct: return "medium"
    if "light" in ct: return "light"
    return "other"

df["roast"] = df["coffee_type"].apply(extract_roast)

print(f"\nShape after cleaning : {df.shape}")
print(df[["price","rating","reviews","price_per_gram","roast"]].describe().round(3))



Shape after cleaning : (1373, 13)
          price    rating    reviews  price_per_gram
count  1373.000  1373.000   1373.000        1373.000
mean     14.184     4.061    449.523           0.032
std       9.466     1.425    885.764           0.100
min       1.000     0.000      0.000           0.001
25%       8.510     4.300     19.000           0.015
50%      12.980     4.600    147.000           0.022
75%      16.400     4.800    613.000           0.034
max      77.090     5.000  15148.000           2.952


In [35]:
# 3. SAVE OUTPUTS PATH
# Use a local folder to save figures
output_dir = "C:/Users/hp/Desktop/your_project/outputs/"
import os
os.makedirs(output_dir, exist_ok=True)

In [82]:
# Example of saving a figure
plt.figure()
sns.histplot(df["rating"], bins=20, kde=True)
plt.title("Rating Distribution")
output_path = os.path.join(output_dir, "rating_distribution.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")


  → Saved: C:/Users/hp/Desktop/your_project/outputs/rating_distribution.png


In [38]:
# 3. EDA VISUALIZATIONS
print("\n" + "="*60)
print("  STEP 3 — EDA PLOTS")
print("="*60)


  STEP 3 — EDA PLOTS


In [39]:
fig, axes = plt.subplots(2, 3, figsize=(17, 10))
fig.suptitle("Walmart Coffee — Exploratory Data Analysis", fontsize=16, fontweight="bold", y=1.01)


Text(0.5, 1.01, 'Walmart Coffee — Exploratory Data Analysis')

In [42]:
# 3a Rating distribution
sns.histplot(df["rating"], bins=20, kde=True, ax=axes[0,0], color=PALETTE[0])
axes[0,0].set_title("Rating Distribution")


Text(0.5, 1.0, 'Rating Distribution')

In [43]:
# 3b Price distribution
sns.histplot(df["price"].clip(upper=50), bins=30, kde=True, ax=axes[0,1], color=PALETTE[1])
axes[0,1].set_title("Price Distribution (clipped $50)")

Text(0.5, 1.0, 'Price Distribution (clipped $50)')

In [44]:
# 3c Reviews log-scale
sns.histplot(df["log_reviews"], bins=30, kde=True, ax=axes[0,2], color=PALETTE[2])
axes[0,2].set_title("log(1+reviews) Distribution")


Text(0.5, 1.0, 'log(1+reviews) Distribution')

In [81]:
# 3d Roast count
roast_counts = df["roast"].value_counts()
roast_counts.plot(kind="bar", ax=axes[1,0], color=PALETTE[:4])
axes[1,0].set_title("Products by Roast Type")
axes[1,0].tick_params(axis="x", rotation=30)


In [79]:
# 3e Price vs Rating scatter
axes[1,1].scatter(df["price"].clip(upper=50), df["rating"],
                  alpha=0.3, c=PALETTE[4], edgecolors="white", linewidths=0.3)
axes[1,1].set_title("Price vs Rating")
axes[1,1].set_xlabel("Price ($)")
axes[1,1].set_ylabel("Rating")


Text(720.5166666666667, 0.5, 'Rating')

In [80]:
# 3f Avg rating by roast
df.groupby("roast")["rating"].mean().sort_values().plot(
    kind="barh", ax=axes[1,2], color=PALETTE[:4])
axes[1,2].set_title("Avg Rating by Roast")

plt.tight_layout()
output_path = os.path.join(output_dir, "1_EDA.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")

  → Saved: C:/Users/hp/Desktop/your_project/outputs/1_EDA.png


In [48]:
# 4. CORRELATION HEATMAP
print("\n" + "="*60)
print("  STEP 4 — CORRELATION HEATMAP")
print("="*60)
num_cols = ["price", "rating", "reviews", "weight_formatted_to_gramms",
            "price_per_gram", "log_reviews"]
corr = df[num_cols].corr()
print(corr.round(3))


  STEP 4 — CORRELATION HEATMAP
                            price  rating  reviews  \
price                       1.000   0.045    0.115   
rating                      0.045   1.000    0.198   
reviews                     0.115   0.198    1.000   
weight_formatted_to_gramms  0.301   0.097    0.166   
price_per_gram              0.180  -0.019   -0.030   
log_reviews                 0.033   0.691    0.558   

                            weight_formatted_to_gramms  price_per_gram  \
price                                            0.301           0.180   
rating                                           0.097          -0.019   
reviews                                          0.166          -0.030   
weight_formatted_to_gramms                       1.000          -0.161   
price_per_gram                                  -0.161           1.000   
log_reviews                                      0.179          -0.056   

                            log_reviews  
price                       

In [76]:
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="YlGn",
            linewidths=0.5, ax=ax, square=True)
ax.set_title("Correlation Matrix — Walmart Coffee", fontsize=13, fontweight="bold")
plt.tight_layout()
output_path = os.path.join(output_dir, "2_Correlation_Heatmap.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")


  → Saved: C:/Users/hp/Desktop/your_project/outputs/2_Correlation_Heatmap.png


In [74]:
# 5. NORMALIZATION
print("\n" + "="*60)
print("  STEP 5 — MIN-MAX NORMALIZATION")
print("="*60)
scaler = MinMaxScaler()
features = ["price", "rating", "reviews", "weight_formatted_to_gramms", "price_per_gram", "log_reviews"]
df_norm = df.copy()
df_norm[features] = scaler.fit_transform(df[features])
print("Normalized (0-1) stats:")
print(df_norm[features].describe().round(3))

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle("Normalized Feature Distributions (Min-Max)", fontsize=14, fontweight="bold")
for i, col in enumerate(features):
    r, c = divmod(i, 3)
    sns.histplot(df_norm[col], bins=25, kde=True, ax=axes[r, c], color=PALETTE[i % len(PALETTE)])
    axes[r, c].set_title(col)
plt.tight_layout()
output_path = os.path.join(output_dir, "3_Normalized_Distributions.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")


  STEP 5 — MIN-MAX NORMALIZATION
Normalized (0-1) stats:
          price    rating   reviews  weight_formatted_to_gramms  \
count  1373.000  1373.000  1373.000                    1373.000   
mean      0.173     0.812     0.030                       0.218   
std       0.124     0.285     0.058                       0.130   
min       0.000     0.000     0.000                       0.000   
25%       0.099     0.860     0.001                       0.117   
50%       0.157     0.920     0.010                       0.218   
75%       0.202     0.960     0.040                       0.298   
max       1.000     1.000     1.000                       1.000   

       price_per_gram  log_reviews  
count        1373.000     1373.000  
mean            0.011        0.470  
std             0.034        0.240  
min             0.000        0.000  
25%             0.005        0.311  
50%             0.007        0.519  
75%             0.011        0.667  
max             1.000        1.000  
  → S

In [73]:
# 6. PAIR PLOT
print("\n" + "="*60)
print("  STEP 6 — PAIR PLOT")
print("="*60)
pair_cols = ["price", "rating", "log_reviews", "price_per_gram", "roast"]
pair_df = df[pair_cols].dropna()
g = sns.pairplot(pair_df, hue="roast", palette=PALETTE[:4],
                 diag_kind="kde", plot_kws={"alpha": 0.4, "s": 20})
g.fig.suptitle("Pair Plot — Key Features by Roast Type", y=1.02, fontsize=14, fontweight="bold")
output_path = os.path.join(output_dir, "4_Pair_Plot.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")



  STEP 6 — PAIR PLOT
  → Saved: C:/Users/hp/Desktop/your_project/outputs/4_Pair_Plot.png
  → Saved: 4_Pair_Plot.png


In [54]:
# PREPARE FEATURES FOR ML
le_roast = LabelEncoder()
df["roast_enc"] = le_roast.fit_transform(df["roast"])

X_reg = df[["weight_formatted_to_gramms", "log_reviews", "roast_enc", "price_per_gram"]].copy()
X_reg.fillna(X_reg.median(), inplace=True)
y_rating = df["rating"].copy()
y_price  = df["price"].fillna(df["price"].median()).copy()
y_popular= df["is_popular"].copy()

# Normalize X
X_scaled = pd.DataFrame(scaler.fit_transform(X_reg), columns=X_reg.columns)

X_tr, X_te, yr_tr, yr_te = train_test_split(X_scaled, y_rating, test_size=0.2, random_state=42)
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_scaled, y_popular, test_size=0.2, random_state=42)
Xpr_tr, Xpr_te, yprice_tr, yprice_te = train_test_split(X_scaled, y_price, test_size=0.2, random_state=42)


In [72]:
# 7. LINEAR REGRESSION — predict rating
print("\n" + "="*60)
print("  STEP 7 — LINEAR REGRESSION  (Predict Rating)")
print("="*60)
lr = LinearRegression()
lr.fit(X_tr, yr_tr)
yr_pred = lr.predict(X_te)
mse_lr = mean_squared_error(yr_te, yr_pred)
r2_lr  = r2_score(yr_te, yr_pred)
print(f"MSE : {mse_lr:.4f}")
print(f"R²  : {r2_lr:.4f}")
print(f"Coef: {dict(zip(X_reg.columns, lr.coef_.round(4)))}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Linear Regression — Predicting Rating", fontsize=13, fontweight="bold")
axes[0].scatter(yr_te, yr_pred, alpha=0.4, color=PALETTE[0], edgecolors="white", linewidths=0.3)
axes[0].plot([yr_te.min(), yr_te.max()], [yr_te.min(), yr_te.max()], "r--", lw=2)
axes[0].set_xlabel("Actual Rating")
axes[0].set_ylabel("Predicted Rating")
axes[0].set_title(f"Actual vs Predicted  R²={r2_lr:.3f}")

residuals = yr_te - yr_pred
sns.histplot(residuals, bins=25, kde=True, ax=axes[1], color=PALETTE[1])
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Residual")
plt.tight_layout()
output_path = os.path.join(output_dir, "5_Linear_Regression.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")



  STEP 7 — LINEAR REGRESSION  (Predict Rating)
MSE : 1.0187
R²  : 0.5305
Coef: {'weight_formatted_to_gramms': np.float64(-0.0912), 'log_reviews': np.float64(3.9985), 'roast_enc': np.float64(-0.4853), 'price_per_gram': np.float64(0.6229)}
  → Saved: C:/Users/hp/Desktop/your_project/outputs/5_Linear_Regression.png


In [71]:
# 8. LOGISTIC REGRESSION — predict is_popular
print("\n" + "="*60)
print("  STEP 8 — LOGISTIC REGRESSION  (Predict Popularity)")
print("="*60)
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(Xp_tr, yp_tr)
yp_pred = log_reg.predict(Xp_te)
acc_log = accuracy_score(yp_te, yp_pred)
print(f"Accuracy : {acc_log:.4f}")
print("\nClassification Report:")
print(classification_report(yp_te, yp_pred, target_names=["Not Popular","Popular"]))

cm = confusion_matrix(yp_te, yp_pred)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Logistic Regression — Predicting Popularity", fontsize=13, fontweight="bold")
sns.heatmap(cm, annot=True, fmt="d", cmap="Greens", ax=axes[0],
            xticklabels=["Not Popular","Popular"],
            yticklabels=["Not Popular","Popular"])
axes[0].set_title(f"Confusion Matrix  Acc={acc_log:.3f}")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")
coef_df = pd.Series(log_reg.coef_[0], index=X_reg.columns)
coef_df.sort_values().plot(kind="barh", ax=axes[1], color=PALETTE[:4])
axes[1].set_title("Feature Coefficients")
axes[1].axvline(0, color="gray", linestyle="--")
plt.tight_layout()
output_path = os.path.join(output_dir, "6_Logistic_Regression.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")


  STEP 8 — LOGISTIC REGRESSION  (Predict Popularity)
Accuracy : 0.9891

Classification Report:
              precision    recall  f1-score   support

 Not Popular       0.99      1.00      0.99       205
     Popular       0.99      0.97      0.98        70

    accuracy                           0.99       275
   macro avg       0.99      0.98      0.99       275
weighted avg       0.99      0.99      0.99       275

  → Saved: C:/Users/hp/Desktop/your_project/outputs/6_Logistic_Regression.png


In [70]:
# 9. RANDOM FOREST — Regressor (price) + Classifier (popular)
print("\n" + "="*60)
print("  STEP 9 — RANDOM FOREST")
print("="*60)

# --- Regressor: predict price
rfr = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rfr.fit(Xpr_tr, yprice_tr)
yprice_pred = rfr.predict(Xpr_te)
mse_rfr = mean_squared_error(yprice_te, yprice_pred)
r2_rfr  = r2_score(yprice_te, yprice_pred)
print(f"[Regressor] Price — MSE: {mse_rfr:.4f}  R²: {r2_rfr:.4f}")

# --- Classifier: predict is_popular
rfc = RandomForestClassifier(n_estimators=150, random_state=42, n_jobs=-1)
rfc.fit(Xp_tr, yp_tr)
yp_pred_rf = rfc.predict(Xp_te)
acc_rf = accuracy_score(yp_te, yp_pred_rf)
print(f"[Classifier] Popularity Accuracy: {acc_rf:.4f}")
print(classification_report(yp_te, yp_pred_rf, target_names=["Not Popular","Popular"]))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Random Forest — Regressor & Classifier", fontsize=13, fontweight="bold")

# Actual vs Predicted price
axes[0].scatter(yprice_te, yprice_pred, alpha=0.4, color=PALETTE[3], edgecolors="white", linewidths=0.3)
axes[0].plot([yprice_te.min(), yprice_te.max()], [yprice_te.min(), yprice_te.max()], "r--")
axes[0].set_title(f"Price: Actual vs Predicted  R²={r2_rfr:.3f}")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")

# Feature importance — regressor
fi_reg = pd.Series(rfr.feature_importances_, index=X_reg.columns).sort_values()
fi_reg.plot(kind="barh", ax=axes[1], color=PALETTE[:4])
axes[1].set_title("Feature Importance (Price Regressor)")

# Feature importance — classifier
fi_cls = pd.Series(rfc.feature_importances_, index=X_reg.columns).sort_values()
fi_cls.plot(kind="barh", ax=axes[2], color=PALETTE[2:6])
axes[2].set_title("Feature Importance (Popularity Classifier)")

plt.tight_layout()
output_path = os.path.join(output_dir, "7_Random_Forest.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")



  STEP 9 — RANDOM FOREST
[Regressor] Price — MSE: 11.5419  R²: 0.8656
[Classifier] Popularity Accuracy: 1.0000
              precision    recall  f1-score   support

 Not Popular       1.00      1.00      1.00       205
     Popular       1.00      1.00      1.00        70

    accuracy                           1.00       275
   macro avg       1.00      1.00      1.00       275
weighted avg       1.00      1.00      1.00       275

  → Saved: C:/Users/hp/Desktop/your_project/outputs/7_Random_Forest.png


In [66]:
# 10. K-MEANS CLUSTERING
print("\n" + "="*60)
print("  STEP 10 — K-MEANS CLUSTERING")
print("="*60)

# Elbow method
X_clust = df_norm[["price", "rating", "log_reviews", "price_per_gram"]].dropna()
inertias, sil_scores = [], []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust, km.labels_))
    print(f"  k={k}  inertia={km.inertia_:.1f}  silhouette={sil_scores[-1]:.4f}")

best_k = K_range[np.argmax(sil_scores)]
print(f"\nBest k by silhouette: {best_k}")

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
km_final.fit(X_clust)
df_clust = df.loc[X_clust.index].copy()
df_clust["cluster"] = km_final.labels_

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f"K-Means Clustering  (k={best_k})", fontsize=13, fontweight="bold")

# Elbow
axes[0].plot(K_range, inertias, "o-", color=PALETTE[0])
axes[0].set_title("Elbow Curve")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")

# Silhouette
axes[1].plot(K_range, sil_scores, "s-", color=PALETTE[4])
axes[1].axvline(best_k, color="red", linestyle="--", label=f"best k={best_k}")
axes[1].set_title("Silhouette Scores")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette")
axes[1].legend()

# Scatter by cluster
sc = axes[2].scatter(df_clust["price"].clip(upper=40), df_clust["rating"],
                     c=df_clust["cluster"], cmap="Set2",
                     alpha=0.5, s=30, edgecolors="white", linewidths=0.3)
axes[2].set_title("Price vs Rating by Cluster")
axes[2].set_xlabel("Price ($)")
axes[2].set_ylabel("Rating")
plt.colorbar(sc, ax=axes[2], label="Cluster")

plt.tight_layout()
output_path = os.path.join(output_dir, "8_KMeans_Clustering.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")
print("\n--- Cluster Profile ---")
print(df_clust.groupby("cluster")[["price","rating","reviews","price_per_gram"]].mean().round(3))



  STEP 10 — K-MEANS CLUSTERING
  k=2  inertia=73.8  silhouette=0.7372
  k=3  inertia=41.7  silhouette=0.4813
  k=4  inertia=32.7  silhouette=0.4603
  k=5  inertia=26.5  silhouette=0.3999
  k=6  inertia=22.0  silhouette=0.4074
  k=7  inertia=18.7  silhouette=0.4223
  k=8  inertia=16.8  silhouette=0.3889
  k=9  inertia=15.4  silhouette=0.3756

Best k by silhouette: 2
  → Saved: C:/Users/hp/Desktop/your_project/outputs/8_KMeans_Clustering.png

--- Cluster Profile ---
          price  rating  reviews  price_per_gram
cluster                                         
0        14.241   4.533  502.172           0.032
1        13.699   0.034    0.181           0.032


In [67]:
# 11. BOX PLOTS BY ROAST — Seaborn v0.14+ safe version
import os

print("\n" + "="*60)
print("  STEP 11 — BOX PLOTS BY ROAST")
print("="*60)

# Drop rows with NaNs in the columns we plot
plot_df = df.dropna(subset=["price", "rating", "log_reviews", "roast"])

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Feature Distribution by Roast Type", fontsize=13, fontweight="bold")

# Loop through features
for ax, col, title in zip(axes, ["price", "rating", "log_reviews"],
                          ["Price ($)", "Rating", "log(1+Reviews)"]):
    # Use 'hue' for palette and set legend=False to avoid warnings
    sns.boxplot(
        data=plot_df,
        x="roast",
        y=col,
        hue="roast",
        palette=PALETTE[:4],
        ax=ax,
        dodge=False,
        showfliers=True,
        legend=False
    )
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=30)

# Adjust layout so suptitle doesn't overlap
plt.subplots_adjust(top=0.85)
plt.tight_layout()

# Save the figure
output_path = os.path.join(output_dir, "9_BoxPlots_by_Roast.png")
plt.savefig(output_path, bbox_inches="tight")
plt.close()
print(f"  → Saved: {output_path}")


  STEP 11 — BOX PLOTS BY ROAST
  → Saved: C:/Users/hp/Desktop/your_project/outputs/9_BoxPlots_by_Roast.png


In [68]:
# 12. SUMMARY TABLE
print("\n" + "="*60)
print("  MODEL PERFORMANCE SUMMARY")
print("="*60)
summary = {
    "Model": ["Linear Regression (Rating)",
              "Logistic Regression (Popular)",
              "RF Regressor (Price)",
              "RF Classifier (Popular)"],
    "Metric": ["R²", "Accuracy", "R²", "Accuracy"],
    "Score":  [f"{r2_lr:.4f}", f"{acc_log:.4f}", f"{r2_rfr:.4f}", f"{acc_rf:.4f}"]
}
print(pd.DataFrame(summary).to_string(index=False))

print("\n✅  All outputs saved to /mnt/user-data/outputs/")


  MODEL PERFORMANCE SUMMARY
                        Model   Metric  Score
   Linear Regression (Rating)       R² 0.5305
Logistic Regression (Popular) Accuracy 0.9891
         RF Regressor (Price)       R² 0.8656
      RF Classifier (Popular) Accuracy 1.0000

✅  All outputs saved to /mnt/user-data/outputs/
